In [16]:
import os
import gc
import time
import numpy as np
import polars as pl
import random
from datetime import timedelta

# Machine Learning
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

# Models
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

In [17]:
def combine_hdd_data():
    input_path = "Data/HardDrive/*/*/*.csv"
    output_path = "Data/HardDrive/Combined_HDD_Data_2025.parquet"
    
    print(f"Scanning 46 GB of CSV files in: {input_path}")
    lazy_df = pl.scan_csv(input_path, infer_schema_length=10000, ignore_errors=True)
    
    print(f"Combining and streaming to Parquet file: {output_path}...")
    start = time.time()
    lazy_df.sink_parquet(output_path)
    end = time.time()
    print(f"Done! Combined dataset saved in {end - start:.1f} seconds.")

#combine_hdd_data()


In [18]:
FEATURES = [
    'smart_200_raw', 'smart_197_raw', 'smart_187_raw', 'smart_5_raw', 
    'smart_196_raw', 'smart_2_raw', 'smart_199_raw', 'smart_12_raw', 
    'smart_9_raw', 'smart_188_raw', 'smart_195_raw', 'smart_190_raw', 
    'smart_241_raw', 'smart_198_raw', 'smart_8_raw'
]

In [19]:
def extract_hdd_sequences(parquet_path, window_size=14, lead_days=7):
    print("Pass 1: Scanning for failures (Lazy Mode - low RAM)...")
    lazy_df = pl.scan_parquet(parquet_path)
    
    # 1. Find all failures without loading the whole dataset
    fails_df = lazy_df.filter(pl.col('failure') == 1).select(['serial_number']).collect()
    failed_serials = fails_df['serial_number'].unique().to_list()
    failed_set = set(failed_serials)
    
    # 2. Get unique healthy serials
    print("Pass 2: Sampling healthy drives...")
    all_serials = lazy_df.select('serial_number').unique().collect()['serial_number'].to_list()
    healthy_serials = [s for s in all_serials if s not in failed_set]
    
    # Sub-sample healthy drives heavily (3:1 ratio) to save RAM
    rng = random.Random(42)
    target_healthy = len(failed_serials) * 3
    if len(healthy_serials) > target_healthy:
        sampled_healthy = set(rng.sample(healthy_serials, target_healthy))
    else:
        sampled_healthy = set(healthy_serials)
        
    serials_to_keep = list(failed_set.union(sampled_healthy))
    
    print(f"Pass 3: Loading data for strictly {len(serials_to_keep)} selected drives...")
    columns_to_keep = ['date', 'serial_number', 'failure'] + FEATURES
    
    # 3. Read ONLY the drives we care about into RAM
    df = (
        lazy_df
        .filter(pl.col('serial_number').is_in(serials_to_keep))
        .select(columns_to_keep)
        .with_columns([pl.col(c).cast(pl.Float32) for c in FEATURES])
        .with_columns(pl.col('date').str.to_date('%Y-%m-%d'))
        .sort(['serial_number', 'date'])
        .collect()
    )
    
    X, y = [], []
    groups = df.group_by('serial_number')
    
    print(f"Extracting {window_size}-day sequences with {lead_days}-day forecast horizon...")
    
    for name, group in groups:
        serial = name[0]
            
        dates = group['date'].to_list()
        vals = group[FEATURES].to_numpy()
        
        if len(dates) < window_size:
            continue
            
        if serial in failed_set:
            fail_date = dates[-1] 
            target_date = fail_date - timedelta(days=lead_days)
            start_date = target_date - timedelta(days=window_size - 1)
            
            try:
                target_idx = dates.index(target_date)
            except ValueError:
                continue
                
            start_idx = target_idx - window_size + 1
            if start_idx < 0:
                continue
                
            if (dates[target_idx] - dates[start_idx]).days != window_size - 1:
                continue
                
            X.append(vals[start_idx : target_idx + 1])
            y.append(1)
            
        else:
            valid_starts = []
            for i in range(len(dates) - window_size + 1):
                if (dates[i + window_size - 1] - dates[i]).days == window_size - 1:
                    valid_starts.append(i)
                    
            if valid_starts:
                start_idx = rng.choice(valid_starts)
                X.append(vals[start_idx : start_idx + window_size])
                y.append(0)
                
    return np.array(X), np.array(y)


In [20]:
def engineer_tree_features(X_3d):
    """ Flattens a 3D sequential array into 2D metrics for XGBoost. """
    last_step = X_3d[:, -1, :]
    mean_val = np.mean(X_3d, axis=1)
    delta_val = X_3d[:, -1, :] - X_3d[:, 0, :]
    max_val = np.max(X_3d, axis=1)
    return np.hstack([last_step, mean_val, delta_val, max_val])

In [21]:
def build_CNN_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model
def build_LSTM_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model
def build_Hybrid_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)
    attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model
def build_GRU_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model
    
def build_ResNet1D_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', 
                      kernel_regularizer=regularizers.l2(0.001))(inputs)
    x = layers.BatchNormalization()(x)
    res = x
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv1D(64, kernel_size=3, padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    res = layers.Conv1D(128, kernel_size=1, padding='same')(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x) 
    x = layers.Conv1D(128, kernel_size=3, padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.5)(x) 
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model
    
def build_TCN_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for dilation_rate in [1, 2, 4, 8]:
        res = layers.Conv1D(64, kernel_size=1, padding='same')(x)
        x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=dilation_rate, 
                          activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x) 
        x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=dilation_rate, 
                          kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Add()([res, x])
        x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_ResNet_Hybrid_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    res = x
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    res = layers.Conv1D(128, kernel_size=1, padding='same')(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)
    attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_Deep_CNN_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2), 
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

In [22]:
parquet_path = "Data/HardDrive/Combined_HDD_Data_2025.parquet"

# Feel free to adjust the window_size and lead_days to whatever you want!
X, y = extract_hdd_sequences(parquet_path, window_size=14, lead_days=7)
print(f"\nExtracted {len(X)} sequences (Failures: {np.sum(y)}, Healthy: {len(y) - np.sum(y)})")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("\nImputing NaNs and Scaling data...")
N_tr, W, F = X_train_raw.shape
N_te = X_test_raw.shape[0]

X_train_flat = X_train_raw.reshape(-1, F)
X_test_flat = X_test_raw.reshape(-1, F)

imputer = SimpleImputer(strategy='median')
X_train_flat = imputer.fit_transform(X_train_flat)
X_test_flat = imputer.transform(X_test_flat)

scaler = StandardScaler()
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)

X_train = X_train_flat.reshape(N_tr, W, F)
X_test = X_test_flat.reshape(N_te, W, F)

# Balance Training Set to exactly 50:50
fail_idx = np.where(y_train == 1)[0]
healthy_idx = np.where(y_train == 0)[0]
target_healthy_count = len(fail_idx)
sampled_healthy_idx = np.random.default_rng(42).choice(healthy_idx, size=target_healthy_count, replace=False)

keep_idx = np.concatenate([fail_idx, sampled_healthy_idx])
np.random.default_rng(42).shuffle(keep_idx)

X_train_balanced = X_train[keep_idx]
y_train_balanced = y_train[keep_idx]
print(f"Balanced Training Set: {len(X_train_balanced)} sequences.")

Pass 1: Scanning for failures (Lazy Mode - low RAM)...
Pass 2: Sampling healthy drives...
Pass 3: Loading data for strictly 17240 selected drives...
Extracting 14-day sequences with 7-day forecast horizon...

Extracted 16311 sequences (Failures: 3438, Healthy: 12873)

Imputing NaNs and Scaling data...
Balanced Training Set: 4812 sequences.


In [ ]:
# Define the models to loop over
models_to_test = {
    # Commented out so they don't run again, saving you time!
#     "XGBoost": "baseline_supervised",
      "LightGBM": "baseline_supervised",
#     "Individual CNN": build_CNN_model,
#     "1D-ResNet": build_ResNet1D_model,
#     "TCN": build_TCN_model,
#     "Individual LSTM": build_LSTM_model,
#     "Individual GRU": build_GRU_model,
#     "Hybrid": build_Hybrid_model,
#     "Deep CNN": build_Deep_CNN_model,
#     "ResNet-Hybrid": build_ResNet_Hybrid_model
}
results = {}
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5,
    restore_best_weights=True
)
for name, model_builder in models_to_test.items():
    print(f"\n--- Training {name} ---")
    start_time = time.time()
    
    if model_builder == "baseline_supervised":
        X_train_tree = engineer_tree_features(X_train_balanced)
        X_test_tree = engineer_tree_features(X_test)
        
        if name == "XGBoost":
            m = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
        elif name == "LightGBM":
            m = lgb.LGBMClassifier(random_state=42, verbose=-1)
            
        m.fit(X_train_tree, y_train_balanced)
        preds = m.predict(X_test_tree)
        
    else:
        # It's a Keras model
        m = model_builder((W, F))
        m.fit(X_train_balanced, y_train_balanced, epochs=20, batch_size=32, verbose=1, validation_split=0.1, callbacks=[early_stop])
        preds_prob = m.predict(X_test, verbose=0)
        preds = (preds_prob > 0.5).astype(int).flatten()
        
    train_time = time.time() - start_time
    score = f1_score(y_test, preds)
    print(f"{name} F1 Score: {score:.4f} (Took {train_time:.2f}s)")
    
    results[name] = {
        "f1_score": score,
        "training_time_seconds": round(train_time, 2)
    }
    
print("FINAL MODEL RESULTS: ")
for k, v in results.items():
    print(f"{k.ljust(20)}: F1 = {v['f1_score']:.4f}")


--- Training XGBoost ---
XGBoost F1 Score: 0.7521 (Took 0.14s)

--- Training LightGBM ---


C:\Users\sabbu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [13:40:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


LightGBM F1 Score: 0.7750 (Took 0.30s)

--- Training Individual CNN ---
Epoch 1/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5757 - val_loss: 0.6033
Epoch 2/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5343 - val_loss: 0.5493
Epoch 3/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5108 - val_loss: 0.5273
Epoch 4/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4920 - val_loss: 0.5138
Epoch 5/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4841 - val_loss: 0.5381
Epoch 6/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4753 - val_loss: 0.5141
Epoch 7/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4663 - val_loss: 0.5086
Epoch 8/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4592 - val_loss: 0.5097
Epoch 9/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4533 - val_loss: 0.4977
Epoch 10/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4459 - val_loss: 0.5072
Epoch 11/20
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4384